# 03 SKU Classification

## 3.1 Business Objective

This notebook classifies SKUs based on sales contribution, demand volume, and demand volatility.

The purpose is to convert transaction data into actionable product groups for inventory planning and warehouse allocation.

Management does not need to review every SKU individually. Instead, SKUs should be grouped into practical categories:

1. High-revenue SKUs that require priority inventory monitoring.
2. High-volume SKUs that require stable replenishment.
3. Long-tail SKUs that may require cautious stocking.
4. Volatile SKUs that may create overstock or stockout risk.

The output of this notebook will support replenishment planning and warehouse allocation in the next stage.

## 3.2 Data Foundation from 01 and 02

This notebook uses `monthly_sku_sales.csv`, which was generated from the cleaned product sales dataset in `01_data_cleaning.ipynb`.

The source data excludes cancellations, returns, invalid sales records, duplicate rows, and non-product transaction lines.

The SQL outputs from `02_sql_business_queries.ipynb` provide supporting business summaries, including top revenue SKUs, top unit-volume SKUs, long-tail SKUs, country demand, and monthly sales trends.

This ensures that SKU classification is based on valid physical product sales rather than mixed transaction records.

## 3.3 Load Monthly SKU Sales

In [13]:
import pandas as pd
import numpy as np
from pathlib import Path

# Define paths
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

processed_dir = project_root / "data" / "processed"
outputs_dir = project_root / "outputs"

monthly_sku_path = processed_dir / "monthly_sku_sales.csv"
sku_master_path = processed_dir / "sku_master.csv"

monthly_sku_sales = pd.read_csv(monthly_sku_path)
sku_master = pd.read_csv(sku_master_path)

print("Monthly SKU sales shape:", monthly_sku_sales.shape)
print("SKU master shape:", sku_master.shape)

monthly_sku_sales.head()

Monthly SKU sales shape: (34020, 7)
SKU master shape: (3917, 7)


,stock_code,description,invoice_month,monthly_units,monthly_revenue,order_count,avg_unit_price
0,10002,INFLATABLE POLITICAL GLOBE,2010-12,251,234.41,30,1.201000
1,10002,INFLATABLE POLITICAL GLOBE,2011-01,340,291.37,21,0.962857
2,10002,INFLATABLE POLITICAL GLOBE,2011-02,52,45.76,7,1.072857
3,10002,INFLATABLE POLITICAL GLOBE,2011-03,28,27.70,8,1.142500
4,10002,INFLATABLE POLITICAL GLOBE,2011-04,189,160.65,5,0.850000


## 3.4 Build SKU-Level Profile

The monthly SKU-level sales table is aggregated into a SKU profile.

This profile summarizes total sales volume, total revenue, active months, average monthly demand, and demand volatility.

These metrics help identify which SKUs are commercially important and which SKUs may create inventory risk.

In [14]:
sku_profile = (
    monthly_sku_sales
    .groupby("stock_code", as_index=False)
    .agg(
        total_units=("monthly_units", "sum"),
        total_revenue=("monthly_revenue", "sum"),
        active_months=("invoice_month", "nunique"),
        avg_monthly_units=("monthly_units", "mean"),
        std_monthly_units=("monthly_units", "std"),
        avg_unit_price=("avg_unit_price", "mean"),
        total_orders=("order_count", "sum")
    )
)

sku_profile = sku_profile.merge(
    sku_master[["stock_code", "description"]],
    on="stock_code",
    how="left"
)

# Fill missing standard deviation for SKUs active in only one month
sku_profile["std_monthly_units"] = sku_profile["std_monthly_units"].fillna(0)

# Demand volatility coefficient
sku_profile["demand_cv"] = np.where(
    sku_profile["avg_monthly_units"] > 0,
    sku_profile["std_monthly_units"] / sku_profile["avg_monthly_units"],
    0
)

print("SKU profile shape:", sku_profile.shape)
sku_profile.head()

SKU profile shape: (3917, 10)


,stock_code,total_units,total_revenue,active_months,avg_monthly_units,std_monthly_units,avg_unit_price,total_orders,description,demand_cv
0,10002,860,759.89,5,172.000000,132.183584,1.045843,71,INFLATABLE POLITICAL GLOBE,0.768509
1,10080,303,119.09,7,43.285714,33.119553,0.455714,22,GROOVY CACTUS INFLATABLE,0.765138
2,10120,192,40.32,10,19.200000,15.454593,0.210000,29,DOGGY RUBBER,0.804927
3,10123C,5,3.25,2,2.500000,2.121320,0.650000,3,HEARTS WRAPPING TAPE,0.848528
4,10124A,16,6.72,4,4.000000,0.816497,0.420000,5,SPOTS ON RED BOOKCOVER TAPE,0.204124


## 3.5 Sales Contribution Thresholds

To classify SKUs, I use percentile-based thresholds.

The top 20% of SKUs by revenue are treated as high-revenue products.  
The top 20% of SKUs by unit sales are treated as high-volume products.  
The bottom 30% of SKUs by unit sales are treated as long-tail products.

This method is simple, transparent, and easy for management to understand.

In [15]:
revenue_80 = sku_profile["total_revenue"].quantile(0.80)
units_80 = sku_profile["total_units"].quantile(0.80)
units_30 = sku_profile["total_units"].quantile(0.30)
cv_75 = sku_profile["demand_cv"].quantile(0.75)

print("Revenue 80th percentile:", round(revenue_80, 2))
print("Units 80th percentile:", round(units_80, 2))
print("Units 30th percentile:", round(units_30, 2))
print("Demand CV 75th percentile:", round(cv_75, 2))

Revenue 80th percentile: 3021.98
Units 80th percentile: 1848.8
Units 30th percentile: 87.0
Demand CV 75th percentile: 1.07


## 3.6 SKU Classification Logic

Each SKU is assigned to one business category based on revenue, sales volume, and demand volatility.

Classification rules:

- High-Revenue Priority: SKUs in the top 20% by revenue.
- High-Turnover Stable: SKUs in the top 20% by units sold with relatively low demand volatility.
- High-Turnover Volatile: SKUs in the top 20% by units sold with high demand volatility.
- Long-Tail: SKUs in the bottom 30% by units sold.
- Regular: all other SKUs.

This classification helps management decide where to focus inventory control and warehouse capacity.

In [16]:
def classify_sku(row):
    if row["total_revenue"] >= revenue_80:
        return "High-Revenue Priority"
    elif row["total_units"] >= units_80 and row["demand_cv"] <= cv_75:
        return "High-Turnover Stable"
    elif row["total_units"] >= units_80 and row["demand_cv"] > cv_75:
        return "High-Turnover Volatile"
    elif row["total_units"] <= units_30:
        return "Long-Tail"
    else:
        return "Regular"

sku_profile["sku_class"] = sku_profile.apply(classify_sku, axis=1)

sku_profile[["stock_code", "description", "total_units", "total_revenue", "demand_cv", "sku_class"]].head()

,stock_code,description,total_units,total_revenue,demand_cv,sku_class
0,10002,INFLATABLE POLITICAL GLOBE,860,759.89,0.768509,Regular
1,10080,GROOVY CACTUS INFLATABLE,303,119.09,0.765138,Regular
2,10120,DOGGY RUBBER,192,40.32,0.804927,Regular
3,10123C,HEARTS WRAPPING TAPE,5,3.25,0.848528,Long-Tail
4,10124A,SPOTS ON RED BOOKCOVER TAPE,16,6.72,0.204124,Long-Tail


## 3.7 Classification Summary

The classification summary shows how many SKUs fall into each group.

This helps management understand the overall product mix and identify whether inventory is concentrated in high-priority SKUs or spread across long-tail products.

In [17]:
classification_summary = (
    sku_profile
    .groupby("sku_class", as_index=False)
    .agg(
        sku_count=("stock_code", "count"),
        total_units=("total_units", "sum"),
        total_revenue=("total_revenue", "sum"),
        avg_demand_cv=("demand_cv", "mean")
    )
    .sort_values("total_revenue", ascending=False)
)

classification_summary["revenue_share"] = (
    classification_summary["total_revenue"] / classification_summary["total_revenue"].sum()
)

classification_summary["unit_share"] = (
    classification_summary["total_units"] / classification_summary["total_units"].sum()
)

classification_summary

,sku_class,sku_count,total_units,total_revenue,avg_demand_cv,revenue_share,unit_share
0,High-Revenue Priority,784,3676817,8093643.960,0.732750,0.788387,0.661112
4,Regular,1684,974028,1580547.140,0.985411,0.153958,0.175136
1,High-Turnover Stable,208,651372,356778.530,0.661710,0.034753,0.117120
3,Long-Tail,1172,30962,137370.863,0.606195,0.013381,0.005567
2,High-Turnover Volatile,69,228387,97743.300,1.349236,0.009521,0.041065


## 3.8 Recommended Inventory Actions

After classifying SKUs, each group is mapped to a practical inventory action.

The goal is not only to label products, but also to translate data into management recommendations.

In [18]:
action_mapping = {
    "High-Revenue Priority": "Prioritize inventory monitoring and avoid stockouts",
    "High-Turnover Stable": "Keep stable local warehouse inventory",
    "High-Turnover Volatile": "Monitor closely and replenish in smaller batches",
    "Long-Tail": "Limit stock and avoid excessive local warehouse space",
    "Regular": "Maintain standard replenishment review"
}

sku_profile["recommended_action"] = sku_profile["sku_class"].map(action_mapping)

sku_profile[
    ["stock_code", "description", "total_units", "total_revenue", "demand_cv", "sku_class", "recommended_action"]
].head(20)

,stock_code,description,total_units,total_revenue,demand_cv,sku_class,recommended_action
0,10002,INFLATABLE POLITICAL GLOBE,860,759.89,0.768509,Regular,Maintain standard replenishment review
1,10080,GROOVY CACTUS INFLATABLE,303,119.09,0.765138,Regular,Maintain standard replenishment review
2,10120,DOGGY RUBBER,192,40.32,0.804927,Regular,Maintain standard replenishment review
3,10123C,HEARTS WRAPPING TAPE,5,3.25,0.848528,Long-Tail,Limit stock and avoid excessive local warehous...
4,10124A,SPOTS ON RED BOOKCOVER TAPE,16,6.72,0.204124,Long-Tail,Limit stock and avoid excessive local warehous...
5,10124G,ARMY CAMO BOOKCOVER TAPE,17,7.14,0.117647,Long-Tail,Limit stock and avoid excessive local warehous...
6,10125,MINI FUNKY DESIGN TAPES,1295,993.99,0.625671,Regular,Maintain standard replenishment review
7,10133,COLOURING PENCILS BROWN TUBE,2856,1539.60,1.058999,High-Turnover Stable,Keep stable local warehouse inventory
8,10135,COLOURING PENCILS BROWN TUBE,2229,2204.89,1.004283,High-Turnover Stable,Keep stable local warehouse inventory
9,11001,ASSTD DESIGN RACING CAR PEN,1615,2389.44,1.050720,Regular,Maintain standard replenishment review


In [19]:
sku_profile.to_csv(outputs_dir / "sku_profile_classification.csv", index=False)
classification_summary.to_csv(outputs_dir / "sku_classification_summary.csv", index=False)

print("Saved output files:")
print("- outputs/sku_profile_classification.csv")
print("- outputs/sku_classification_summary.csv")

Saved output files:
- outputs/sku_profile_classification.csv
- outputs/sku_classification_summary.csv


## 3.9 Management Implication

The SKU classification output shows that product-level sales contribution is highly concentrated.

In this analysis, 784 High-Revenue Priority SKUs contribute approximately 78.8% of total revenue and 66.1% of total units sold. In contrast, 1,172 Long-Tail SKUs contribute approximately 1.3% of revenue and 0.6% of units sold.

This suggests that inventory and warehouse resources should not be allocated evenly across all SKUs.

High-Revenue Priority SKUs should receive closer replenishment monitoring because stockouts may directly affect sales performance. Long-Tail SKUs should be stocked cautiously because they may occupy warehouse space while contributing limited revenue.

This classification provides the foundation for the next step: calculating replenishment needs, inventory risk, and warehouse allocation strategies.

## 3.10 Final Summary Code Cell

In [20]:
print("===== 03 SKU Classification Summary =====")
print("SKU profile shape:", sku_profile.shape)
print("Classification summary shape:", classification_summary.shape)
print("\nSKU classes:")
print(classification_summary[["sku_class", "sku_count", "revenue_share", "unit_share"]])
print("\nOutput files:")
print("- outputs/sku_profile_classification.csv")
print("- outputs/sku_classification_summary.csv")

===== 03 SKU Classification Summary =====
SKU profile shape: (3917, 12)
Classification summary shape: (5, 7)

SKU classes:
                sku_class  sku_count  revenue_share  unit_share
0   High-Revenue Priority        784       0.788387    0.661112
4                 Regular       1684       0.153958    0.175136
1    High-Turnover Stable        208       0.034753    0.117120
3               Long-Tail       1172       0.013381    0.005567
2  High-Turnover Volatile         69       0.009521    0.041065

Output files:
- outputs/sku_profile_classification.csv
- outputs/sku_classification_summary.csv
